In [5]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys = ON;")


drop_sql = """
DROP TABLE IF EXISTS hw2_genre_features_online;
DROP TABLE IF EXISTS hw2_movie_features_online;
DROP TABLE IF EXISTS hw2_user_features_online;
DROP TABLE IF EXISTS hw2_training_events;
DROP TABLE IF EXISTS hw2_genre_features_offline;
DROP TABLE IF EXISTS hw2_movie_features_offline;
DROP TABLE IF EXISTS hw2_user_features_offline;
DROP TABLE IF EXISTS hw2_ratings;
DROP TABLE IF EXISTS hw2_movies;
DROP TABLE IF EXISTS hw2_genres;
DROP TABLE IF EXISTS hw2_users;
"""

conn.executescript(drop_sql)


conn.executescript("""
CREATE TABLE hw2_users (
    user_id INTEGER PRIMARY KEY,
    created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE hw2_genres (
    genre_id INTEGER PRIMARY KEY,
    genre_name TEXT NOT NULL UNIQUE
);

CREATE TABLE hw2_movies (
    movie_id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    release_year INTEGER,
    created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
    CHECK (release_year IS NULL OR release_year >= 1888)
);


CREATE TABLE hw2_ratings (
    rating_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER NOT NULL,
    movie_id INTEGER NOT NULL,
    rating REAL NOT NULL CHECK (rating >= 0.5 AND rating <= 5.0),
    rating_ts TEXT NOT NULL,
    UNIQUE (user_id, movie_id),
    FOREIGN KEY (user_id)
        REFERENCES hw2_users(user_id) ON DELETE CASCADE,
    FOREIGN KEY (movie_id)
        REFERENCES hw2_movies(movie_id) ON DELETE CASCADE
);

CREATE INDEX hw2_idx_ratings_user_ts
    ON hw2_ratings(user_id, rating_ts);

CREATE INDEX hw2_idx_ratings_movie_ts
    ON hw2_ratings(movie_id, rating_ts);


CREATE TABLE hw2_user_features_offline (
    user_id INTEGER NOT NULL,
    valid_from TEXT NOT NULL,
    valid_to TEXT,
    avg_rating REAL,
    ratings_count INTEGER NOT NULL DEFAULT 0,
    updated_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY (user_id, valid_from),
    FOREIGN KEY (user_id)
        REFERENCES hw2_users(user_id) ON DELETE CASCADE,
    CHECK (valid_to IS NULL OR valid_to > valid_from),
    CHECK (avg_rating IS NULL OR
           (avg_rating >= 0.5 AND avg_rating <= 5.0)),
    CHECK (ratings_count >= 0)
);

CREATE INDEX hw2_idx_user_features_asof
    ON hw2_user_features_offline(user_id, valid_from, valid_to);


CREATE TABLE hw2_movie_features_offline (
    movie_id INTEGER NOT NULL,
    valid_from TEXT NOT NULL,
    valid_to TEXT,
    avg_rating REAL,
    ratings_count INTEGER NOT NULL DEFAULT 0,
    updated_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY (movie_id, valid_from),
    FOREIGN KEY (movie_id)
        REFERENCES hw2_movies(movie_id) ON DELETE CASCADE,
    CHECK (valid_to IS NULL OR valid_to > valid_from),
    CHECK (avg_rating IS NULL OR
           (avg_rating >= 0.5 AND avg_rating <= 5.0)),
    CHECK (ratings_count >= 0)
);

CREATE INDEX hw2_idx_movie_features_asof
    ON hw2_movie_features_offline(movie_id, valid_from, valid_to);


CREATE TABLE hw2_genre_features_offline (
    genre_id INTEGER NOT NULL,
    valid_from TEXT NOT NULL,
    valid_to TEXT,
    avg_rating REAL,
    ratings_count INTEGER NOT NULL DEFAULT 0,
    updated_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY (genre_id, valid_from),
    FOREIGN KEY (genre_id)
        REFERENCES hw2_genres(genre_id) ON DELETE CASCADE,
    CHECK (valid_to IS NULL OR valid_to > valid_from),
    CHECK (avg_rating IS NULL OR
           (avg_rating >= 0.5 AND avg_rating <= 5.0)),
    CHECK (ratings_count >= 0)
);

CREATE INDEX hw2_idx_genre_features_asof
    ON hw2_genre_features_offline(genre_id, valid_from, valid_to);


CREATE TABLE hw2_training_events (
    event_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER NOT NULL,
    movie_id INTEGER NOT NULL,
    prediction_ts TEXT NOT NULL,
    label_horizon_end TEXT NOT NULL,
    liked_in_horizon INTEGER NOT NULL
        CHECK (liked_in_horizon IN (0, 1)),
    features_version TEXT NOT NULL,
    created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (user_id)
        REFERENCES hw2_users(user_id) ON DELETE CASCADE,
    FOREIGN KEY (movie_id)
        REFERENCES hw2_movies(movie_id) ON DELETE CASCADE,
    CHECK (label_horizon_end > prediction_ts)
);

CREATE INDEX hw2_idx_training_user_ts
    ON hw2_training_events(user_id, prediction_ts);

CREATE INDEX hw2_idx_training_movie_ts
    ON hw2_training_events(movie_id, prediction_ts);


CREATE TABLE hw2_user_features_online (
    user_id INTEGER PRIMARY KEY,
    avg_rating REAL,
    ratings_count INTEGER NOT NULL DEFAULT 0,
    feature_ts TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
    updated_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (user_id)
        REFERENCES hw2_users(user_id) ON DELETE CASCADE,
    CHECK (avg_rating IS NULL OR
           (avg_rating >= 0.5 AND avg_rating <= 5.0)),
    CHECK (ratings_count >= 0)
);


CREATE TABLE hw2_movie_features_online (
    movie_id INTEGER PRIMARY KEY,
    avg_rating REAL,
    ratings_count INTEGER NOT NULL DEFAULT 0,
    feature_ts TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
    updated_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (movie_id)
        REFERENCES hw2_movies(movie_id) ON DELETE CASCADE,
    CHECK (avg_rating IS NULL OR
           (avg_rating >= 0.5 AND avg_rating <= 5.0)),
    CHECK (ratings_count >= 0)
);


CREATE TABLE hw2_genre_features_online (
    genre_id INTEGER PRIMARY KEY,
    avg_rating REAL,
    ratings_count INTEGER NOT NULL DEFAULT 0,
    feature_ts TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
    updated_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (genre_id)
        REFERENCES hw2_genres(genre_id) ON DELETE CASCADE,
    CHECK (avg_rating IS NULL OR
           (avg_rating >= 0.5 AND avg_rating <= 5.0)),
    CHECK (ratings_count >= 0)
);
""")


tables = conn.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
      AND name LIKE 'hw2_%'
    ORDER BY name;
""").fetchall()

print("=" * 55)
print("MOVIELENS FEATURE STORE DATABASE")
print("=" * 55)
print("Tables created:", len(tables))
print()

for i, (table_name,) in enumerate(tables, start=1):
    print(f"{i}. {table_name}")


import pandas as pd
from IPython.display import display, Markdown

tables = pd.read_sql_query("""
    SELECT name AS table_name
    FROM sqlite_master
    WHERE type = 'table'
      AND name LIKE 'hw2_%'
    ORDER BY name;
""", conn)

display(Markdown("## List of MovieLens Feature Store tables"))
display(tables)

print(f"Total tables: {len(tables)}")

for table_name in tables["table_name"]:
    display(Markdown(f"### Table: `{table_name}`"))

    columns = conn.execute(
        f'PRAGMA table_info("{table_name}")'
    ).fetchall()

    structure = pd.DataFrame(
        columns,
        columns=[
            "cid", "column_name", "data_type",
            "not_null", "default_value", "primary_key"
        ]
    )

    structure["not_null"] = structure["not_null"].map(
        {0: "No", 1: "Yes"}
    )
    structure["primary_key"] = structure["primary_key"].map(
        {0: "No", 1: "Yes"}
    )

    display(structure[
        ["column_name", "data_type", "not_null",
         "default_value", "primary_key"]
    ])

    foreign_keys = conn.execute(
        f'PRAGMA foreign_key_list("{table_name}")'
    ).fetchall()

    if foreign_keys:
        fk_df = pd.DataFrame(
            foreign_keys,
            columns=[
                "id", "seq", "referenced_table",
                "from_column", "to_column",
                "on_update", "on_delete", "match"
            ]
        )

        display(Markdown("**Foreign Keys (FOREIGN KEY):**"))
        display(fk_df[
            ["from_column", "referenced_table",
             "to_column", "on_delete"]
        ])
    else:
        display(Markdown("*There are no external keys.*"))


errors = conn.execute("PRAGMA foreign_key_check").fetchall()

display(Markdown("## Result of the check"))
if len(tables) == 11 and not errors:
    print("Created 11 tables. No foreign key errors.")
else:
    print(f"Found tables: {len(tables)}")
    print("Foreign key errors:", errors)

MOVIELENS FEATURE STORE DATABASE
Tables created: 11

1. hw2_genre_features_offline
2. hw2_genre_features_online
3. hw2_genres
4. hw2_movie_features_offline
5. hw2_movie_features_online
6. hw2_movies
7. hw2_ratings
8. hw2_training_events
9. hw2_user_features_offline
10. hw2_user_features_online
11. hw2_users


## List of MovieLens Feature Store tables

,table_name
0,hw2_genre_features_offline
1,hw2_genre_features_online
2,hw2_genres
3,hw2_movie_features_offline
4,hw2_movie_features_online
5,hw2_movies
6,hw2_ratings
7,hw2_training_events
8,hw2_user_features_offline
9,hw2_user_features_online


Total tables: 11


### Table: `hw2_genre_features_offline`

...

### Table: `hw2_users`

,column_name,data_type,not_null,default_value,primary_key
0,user_id,INTEGER,No,None,Yes
1,created_at,TEXT,Yes,CURRENT_TIMESTAMP,No


*There are no external keys.*

## Result of the check

Created 11 tables. No foreign key errors.


Нормалізація та денормалізація схеми MovieLens Feature Store

Основні таблиці "hw2_users", "hw2_movies" і "hw2_genres" відповідають третій нормальній формі (3НФ), оскільки кожна неключова властивість залежить від первинного ключа, усього ключа та не залежить транзитивно від інших неключових атрибутів. Таблиця "hw2_ratings" також відповідає 3НФ: оцінка та час її створення залежать від запису оцінки, а комбінація "user_id" і "movie_id" є унікальною. Історичні offline-таблиці ознак використовують складений первинний ключ із "entity_id" та "valid_from", тому значення ознак визначаються конкретною сутністю та початком періоду дії. Таблиця "hw2_training_events" зберігає окремі події навчання.

Під час декомпозиції враховано функціональні залежності "user_id → created_at", "movie_id → title, release_year", "genre_id → genre_name", та залежність оцінки від користувача й фільму. Зв’язки між таблицями реалізовано через зовнішні ключі.

Online-таблиці "hw2_user_features_online", "hw2_movie_features_online" і "hw2_genre_features_online" навмисно зберігають попередньо обчислені агрегати, наприклад середню оцінку та кількість оцінок. Це часткова денормалізація, орієнтована на lookup за "entity_id", низьку latency та зменшення кількості "JOIN" під час отримання ознак для рекомендацій.

Ризик полягає в тому, що online-значення можуть застаріти або не збігатися з offline-джерелом. Для синхронізації можна періодично або потоково перераховувати ознаки з offline-таблиць і виконувати "UPSERT" в online-таблиці. Час оновлення та версія ознак допомагають контролювати актуальність даних.